In [3]:
# ============================================================
# LEVIR-Ship YOLO26s -> Sentinel-2 Ship Fine-tuning
# data.yaml 자동 생성 포함
# ============================================================

!pip install -q -U ultralytics

from ultralytics import YOLO
from google.colab import drive
from pathlib import Path
import torch
import yaml


# ============================================================
# 1. Google Drive
# ============================================================

drive.mount(
    "/content/drive"
)


# ============================================================
# 2. Paths
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/LEVIR_Ship_YOLO26"
)

PRETRAINED_MODEL = (
    BASE_DIR
    / "YOLO26s_LEVIR_Ship"
    / "weights"
    / "best.pt"
)

DATASET_DIR = (
    BASE_DIR
    / "Sentinel2_Ship_Dataset"
)

DATA_YAML = (
    DATASET_DIR
    / "data.yaml"
)

PROJECT_DIR = (
    BASE_DIR
    / "Sentinel2_Finetune"
)

RUN_NAME = "LEVIR_to_Sentinel2"


# ============================================================
# 3. Dataset structure check
# ============================================================

print("=" * 60)
print("DATASET CHECK")
print("=" * 60)

required_dirs = [

    DATASET_DIR / "train" / "images",
    DATASET_DIR / "train" / "labels",

    DATASET_DIR / "valid" / "images",
    DATASET_DIR / "valid" / "labels",

    DATASET_DIR / "test" / "images",
    DATASET_DIR / "test" / "labels"
]


for p in required_dirs:

    print(
        p,
        "->",
        p.exists()
    )

    assert p.exists(), (
        f"Missing directory: {p}"
    )


assert PRETRAINED_MODEL.exists(), (
    f"LEVIR best.pt not found:\n"
    f"{PRETRAINED_MODEL}"
)


# ============================================================
# 4. Create data.yaml automatically
# ============================================================

yaml_data = {

    "path": str(DATASET_DIR),

    "train": "train/images",

    "val": "valid/images",

    "test": "test/images",

    "names": {
        0: "ship"
    },

    "nc": 1
}


with open(
    DATA_YAML,
    "w"
) as f:

    yaml.safe_dump(
        yaml_data,
        f,
        sort_keys=False
    )


print()
print("=" * 60)
print("DATA.YAML CREATED")
print("=" * 60)

print(
    DATA_YAML.read_text()
)


# ============================================================
# 5. Dataset statistics
# ============================================================

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff"
}


def count_images(folder):

    return sum(

        1

        for p in folder.iterdir()

        if p.suffix.lower()
        in image_extensions
    )


train_images = count_images(
    DATASET_DIR / "train" / "images"
)

val_images = count_images(
    DATASET_DIR / "valid" / "images"
)

test_images = count_images(
    DATASET_DIR / "test" / "images"
)


train_labels = len(
    list(
        (
            DATASET_DIR /
            "train" /
            "labels"
        ).glob("*.txt")
    )
)

val_labels = len(
    list(
        (
            DATASET_DIR /
            "valid" /
            "labels"
        ).glob("*.txt")
    )
)

test_labels = len(
    list(
        (
            DATASET_DIR /
            "test" /
            "labels"
        ).glob("*.txt")
    )
)


print()
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print(
    "Train:",
    train_images,
    "images /",
    train_labels,
    "labels"
)

print(
    "Valid:",
    val_images,
    "images /",
    val_labels,
    "labels"
)

print(
    "Test :",
    test_images,
    "images /",
    test_labels,
    "labels"
)


# ============================================================
# 6. GPU
# ============================================================

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print()
print("=" * 60)
print("DEVICE")
print("=" * 60)

print(
    "CUDA:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 7. Load LEVIR-Ship best.pt
# ============================================================

print()
print("=" * 60)
print("LOAD LEVIR MODEL")
print("=" * 60)

print(
    PRETRAINED_MODEL
)


model = YOLO(
    str(PRETRAINED_MODEL)
)


# ============================================================
# 8. Fine-tuning
# ============================================================

print()
print("=" * 60)
print("START SENTINEL-2 FINE-TUNING")
print("=" * 60)


results = model.train(

    # Sentinel-2 dataset
    data=str(DATA_YAML),

    # Fine-tuning
    epochs=30,

    # LEVIR 학습과 동일하게 640
    imgsz=640,

    batch=16,

    device=DEVICE,

    workers=2,

    # LEVIR best.pt weight 사용
    pretrained=True,

    optimizer="auto",

    # 작은 데이터셋이므로 early stopping
    patience=10,

    # 결과 저장
    project=str(PROJECT_DIR),

    name=RUN_NAME,

    exist_ok=True,

    plots=True
)


# ============================================================
# 9. Find fine-tuned best.pt
# ============================================================

FINETUNED_MODEL = (

    PROJECT_DIR
    / RUN_NAME
    / "weights"
    / "best.pt"
)


print()
print("=" * 60)
print("TRAINING FINISHED")
print("=" * 60)

print(
    "Fine-tuned best.pt:"
)

print(
    FINETUNED_MODEL
)

print(
    "Exists:",
    FINETUNED_MODEL.exists()
)


# ============================================================
# 10. Validation
# ============================================================

if FINETUNED_MODEL.exists():

    print()
    print("=" * 60)
    print("VALIDATION")
    print("=" * 60)

    ft_model = YOLO(
        str(FINETUNED_MODEL)
    )

    metrics = ft_model.val(

        data=str(DATA_YAML),

        split="val",

        imgsz=640,

        batch=16,

        device=DEVICE
    )


# ============================================================
# 11. Done
# ============================================================

print()
print("=" * 60)
print("DONE")
print("=" * 60)

print(
    "Busan Sentinel-2에 사용할 모델:"
)

print(
    FINETUNED_MODEL
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET CHECK
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/train/images -> True
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/train/labels -> True
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/valid/images -> True
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/valid/labels -> True
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/test/images -> True
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/test/labels -> True

DATA.YAML CREATED
path: /content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset
train: train/images
val: valid/images
test: test/images
names:
  0: ship
nc: 1


DATASET STATISTICS
Train: 582 images / 582 labels
Valid: 126 images / 126 labels
Test : 126 images / 126 labels

DEVICE
CUDA: True
GPU: Tesla T4

LOAD LEVIR MODEL
/content/drive/M